# CV Masterclass 02: Architectural Evolution

Welcome to Notebook 02. Knowing how a convolution works is the first step. Knowing how to stack 152 convolutions on top of each other without destroying the network is the second step.

We will explore the paradigm shift from building networks 'Wider and Shallower' to 'Narrower and Deeper', culminating in Kaiming He's Nobel-worthy ResNet architecture.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer involves mathematical degradation:

> *"A $1\times1$ convolution seems physically useless—it just multiplies a pixel by a number. Why are they the most critical computational component in deep architectures like Inception and ResNet?"*

---

## Table of Contents
1. **The VGG Era:** The power of depth and the parameter explosion.
2. **The Inception Module:** Multi-Scale Parallel Processing and the $1\times1$ Bottleneck.
3. **The Degradation Problem:** Why 56 layers performs worse than 20 layers.
4. **Residual Networks (ResNet):** Skip Connections and Bottlenecks.
5. **Squeeze-and-Excitation (SENet):** Channel attention mechanisms.
6. **DenseNet:** All-to-All Skip Connections and Mem checks.
7. **EfficientNet & Compound Scaling:** The empirical sizing law.

## 1. The VGG Era & The Parameter Explosion

In 2014, VGG-16 formalized: Always use $3\times3$ convolutions stacked. Never use massive $5\times5$ or $7\times7$ single layers.

However, as feature maps pass through pooling layers and shrink in spatial dimensions (e.g., $112\times112 \rightarrow 56\times56$), architects simultaneously *double* the number of channels (e.g., $64 \rightarrow 128$) to preserve the information capacity.

A standard VGG-16 convolution deep in the network requires mapping $512$ input channels to $512$ output channels using a $3\times3$ kernel.
**The Cost:** $3 \times 3 \times 512 \times 512 = 2.3 \text{ Million}$ parameters for a *single layer*. This caused massive memory caching issues and terrifying computational overhead.

### ⚠️ Common Pitfalls
*   **Arbitrary Scaling:** Blindly doubling channels after every pooling layer without considering the parameter multiplier $C_{in} \times C_{out}$ can easily blow out edge GPU memory limits.

## 2. The Inception Module (Multi-Scale Parallel Processing)

GoogLeNet asked a fundamental question: *What if we don't know the exact right kernel size for the feature at this depth?* Should we look for tight $1\times1$ patterns, moderate $3\times3$ patterns, or broad $5\times5$ spatial patterns?

**The Answer:** Apply $1\times1$, $3\times3$, AND $5\times5$ convolutions in parallel and concatenate the results along the channel dimension. The network learns which filter scale has the strongest resonance.

### The Problem: Parameter Explosion
If $C_{in} = 256$, and we want to output $C_{out} = 128$ for both the $3\times3$ and $5\times5$ branches directly:
*   $3\times3$ Conv params: $256 \times 128 \times 3 \times 3 = 294,912$
*   $5\times5$ Conv params: $256 \times 128 \times 5 \times 5 = 819,200$
*   Total for these two branches: **$1,114,112$ parameters**.

### The Solution: The $1\times1$ Bottleneck Prefix
We introduce the $1\times1$ Convolution. A $1\times1$ convolution performs no spatial filtering. It only acts across the **channel dimension** to perform computationally cheap cross-channel dimensionality reduction.

If we insert a $1\times1$ bottleneck to compress $C_{in}$ from $256$ to $64$ *before* the spatial filters:
*   **$3\times3$ Branch with bottleneck:**
    *   $1\times1$ projection ($256 \rightarrow 64$): $256 \times 64 \times 1 \times 1 = 16,384$
    *   $3\times3$ explicit ($64 \rightarrow 128$): $64 \times 128 \times 3 \times 3 = 73,728$
    *   Subtotal: $90,112$ parameters.
*   **$5\times5$ Branch with bottleneck:**
    *   $1\times1$ projection ($256 \rightarrow 32$): $256 \times 32 \times 1 \times 1 = 8,192$
    *   $5\times5$ explicit ($32 \rightarrow 128$): $32 \times 128 \times 5 \times 5 = 102,400$
    *   Subtotal: $110,592$ parameters.

**New Total Cost:** $200,704$ parameters.
By using a $1\times1$ prefix, the computations are reduced **by nearly $80\%$**, mathematically enabling multi-scale branches!

In [ ]:
import torch
import torch.nn as nn

class InceptionModule(nn.Module):
    def __init__(self, in_channels):
        super(InceptionModule, self).__init__()
        
        # Branch 1: 1x1 conv
        self.branch1 = nn.Conv2d(in_channels, 64, kernel_size=1)
        
        # Branch 2: 1x1 conv -> 3x3 conv
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, 48, kernel_size=1),
            nn.Conv2d(48, 128, kernel_size=3, padding=1) # Padding=1 to preserve HxW
        )
        
        # Branch 3: 1x1 conv -> 5x5 conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=1),
            nn.Conv2d(16, 32, kernel_size=5, padding=2) # Padding=2 to preserve HxW
        )
        
        # Branch 4: MaxPool2d -> 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, 32, kernel_size=1)
        )

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)
        
        # Concatenate along the channel dimension (Dim 1)
        # Expected output channels = 64 + 128 + 32 + 32 = 256
        return torch.cat([out1, out2, out3, out4], 1)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Input tensor: Batch of 2, 192 channels, 28x28 image
dummy_inception_in = torch.randn(2, 192, 28, 28)

inception = InceptionModule(in_channels=192)
inception_out = inception(dummy_inception_in)

print(f"Inception Input  Shape: {dummy_inception_in.shape}")
print(f"Inception Output Shape: {inception_out.shape}")
assert inception_out.shape == (2, 256, 28, 28), "Inception module failed to preserve spatial dimensions or concatenate correctly!"


### ⚠️ Common Pitfalls
*   **Forgetting padding:** If you don't pad the $3\times3$ branch with $p=1$ and the $5\times5$ branch with $p=2$, the resulting tensors will have different spatial dimensions and `torch.cat` will crash.

## 3. The Degradation Problem

With bottlenecks making deep networks computationally cheap, and Batch Normalization solving the Vanishing Gradient problem, researchers tried to train 100-layer networks.

They completely failed. A 56-layer network had significantly higher Training Error and Test Error than a 20-layer network.

This was not Overfitting. Overfitting generates low training error and high test error. A 56-layer network was simply incapable of optimizing the training set.

**The Paradox:** Let Network A be 20 layers deep. Let Network B be 56 layers deep. Technically, Network B should be able to exactly replicate Network A by just setting its first 20 layers to the exact same weights, and configuring layers 21-56 to physically do nothing (an Identity Mapping: $f(x) = x$). 
The fact that the optimizer failed to find this simple Identity Mapping proved that deep non-linear layers naturally struggle to mathematically achieve absolute identity ($0.0$ change).

### ⚠️ Common Pitfalls
*   **Conflating Degradation with Vanishing Gradients:** Vanishing gradients prevent layers from learning because gradients reach exactly zero. By 2015, Batch Normalization largely cured Vanishing Gradients. Degradation is a *geometry constraint*—the non-linear transformations are actively hostile to preserving identity.


## 4. Residual Networks (ResNet)

Kaiming He solved the Degradation Problem by physically hardwiring the Identity Mapping into the architecture via the **Skip Connection** (or Shortcut Connection).

Instead of forcing layers to learn the final output $H(x)$, ResNet forces them to learn the **Residual** $F(x) = H(x) - x$. The final output is then explicitly calculated as $F(x) + x$.

If the network determines a layer is unnecessary, it easily shrinks the weights of $F(x) \rightarrow 0$. Thus the output evaluates to exactly $(0) + x = x$. 

### The Bottleneck Block & Projection Shortcuts
In real applications, the input identity $x$ and the processed residual $F(x)$ might not have the same shape!
1. Spatial dimensions may shrink (e.g. `stride=2`).
2. Channel counts may expand.

When shapes mismatch, we cannot perform strict addition $F(x) + x$. We must introduce a **Projection Shortcut**, using a $1\times1$ convolution with appropriate striding to instantly force $x$ into the correct shape.
Additionally, very deep ResNets use a **Bottleneck Architecture** ($1\times1 \rightarrow 3\times3 \rightarrow 1\times1$) to keep parameter sizes minimal.


In [ ]:
class ResNetBottleneck(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, out_channels, stride=1):
        super(ResNetBottleneck, self).__init__()
        
        # F(x) non-linear residual path
        self.residual = nn.Sequential(
            # 1x1 Compress
            nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            
            # 3x3 Spatial (with potential stride shrinking)
            nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            
            # 1x1 Expand
            nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        
        # Projection Shortcut
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                # Use a 1x1 to mathematically match dimensions and/or downsample!
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            
        self.final_relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        fx = self.residual(x)
        out = fx + identity
        return self.final_relu(out)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_res_in = torch.randn(2, 256, 56, 56)

# Applying a stride=2 bottleneck. Downsamples 56x56 -> 28x28, Expands 256 -> 1024 channels.
bottleneck = ResNetBottleneck(in_channels=256, bottleneck_channels=128, out_channels=1024, stride=2)
res_out = bottleneck(dummy_res_in)

print(f"ResNet Input Shape:  {dummy_res_in.shape}")
print(f"ResNet Output Shape: {res_out.shape}")
assert res_out.shape == (2, 1024, 28, 28), "ResNet projection shortcut failed!"


### Pre-Activation ResNet (He et al. 2016)

The original ResNet applies Batch Normalization (BN) and ReLU **AFTER** the residual addition: `ReLU(F(x) + x)`. 
While this works well, mathematically the identity path is repeatedly manipulated by the post-addition ReLU, meaning the path isn't a "pure" unbroken gradient superhighway.

To solve this, He et al. proposed the **Pre-Activation ResNet** (ResNet v2). In this variant:
- BN and ReLU are applied **BEFORE** each convolution operation.
- The residual formulation becomes: `F(x) = W2 * ReLU(BN(W1 * ReLU(BN(x))))`
- The shortcut is now a pure mathematical identity: `out = F(x) + x` (without a trailing ReLU!).

This allows gradients to flow completely unobstructed through the skip path.

In [ ]:
class PreActBottleneck(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, out_channels, stride=1):
        super(PreActBottleneck, self).__init__()
        
        # Pre-activation applies BN and ReLU BEFORE the convolutions
        self.residual = nn.Sequential(
            # Pre-act 1
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, stride=1, bias=False),
            
            # Pre-act 2
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            
            # Pre-act 3
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, stride=1, bias=False)
        )
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            # If spatial dimensions change, the shortcut needs a projection to match shapes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
            )

    def forward(self, x):
        # The shortcut is now a pure identity path (no final ReLU!)
        out = self.residual(x) + self.shortcut(x)
        return out

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_preact_in = torch.randn(2, 256, 56, 56)

preact_bottleneck = PreActBottleneck(in_channels=256, bottleneck_channels=128, out_channels=1024, stride=2)
preact_out = preact_bottleneck(dummy_preact_in)

print(f"Pre-Act ResNet Input Shape:  {dummy_preact_in.shape}")
print(f"Pre-Act ResNet Output Shape: {preact_out.shape}")
assert preact_out.shape == (2, 1024, 28, 28), "Pre-Act ResNet projection shortcut failed!"


### Visualizing Degradation
Below is a synthetic plot structurally mirroring the empirical proof recorded in the original ResNet paper. Even with identical capacity, non-linear layers without bypass connections inevitably plateau and drift.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Synthetic Training Loss curves over 100 epochs
epochs = np.arange(1, 101)

# Plain 20 plateauing
plain_20_loss = 2.5 * np.exp(-0.03 * epochs) + 0.9 + np.random.normal(0, 0.05, 100)

# ResNet 20 smoothly optimizing directly toward 0.0
resnet_20_loss = 2.6 * np.exp(-0.08 * epochs) + 0.2 + np.random.normal(0, 0.02, 100)

plt.figure(figsize=(8, 5))
plt.plot(epochs, plain_20_loss, label="Plain-20 Network", color='red', alpha=0.8)
plt.plot(epochs, resnet_20_loss, label="ResNet-20 Network", color='blue', alpha=0.8)

plt.title("The Degradation Cure (Synthetic Proxy)")
plt.xlabel("Epoch")
plt.ylabel("Training Error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### ⚠️ Common Pitfalls
*   **ReLU Placement:** Ensure the final `ReLU` occurs *after* the `+` operator! If `ReLU` is applied directly to $F(x)$, then $F(x)$ can never be negative. It loses the ability to dynamically subtract from the identity mapping!

## 5. Squeeze-and-Excitation (SENet)

We've learned that standard convolutions sum up weights uniformly across every channel. But not all feature channels are equally useful at any given moment. A channel detecting "dog eyes" shouldn't hold equal mathematical weight as the channel detecting "tree leaves" if we are staring at a dog.

**Channel Attention:** SENet introduced a small sub-network inside a functional block that auto-calibrates feature importance:
1. **Squeeze:** Globally Average Pool the $H \times W$ feature map into a single $1 \times 1$ scalar per channel.
2. **Excitation:** Pass this scalar vector deep into a small 2-layer Fully-Connected network (with a bottleneck parameter reduction to save efficiency, e.g. `reduction_ratio=16`).
3. **Reweight:** Pass the output through a Sigmoid (0 to 1). Scale the original massive feature map channels by these respective scalar weights!

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        # Squeeze: spatial pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        
        # Excitation: 2 FC layers (bottlenecked)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            # Sigmoid maps outputs to a strict (0.0 to 1.0) multiplier
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Step 1: Squeeze (b, c, h, w) -> (b, c, 1, 1) -> (b, c)
        y = self.avg_pool(x).view(b, c)
        
        # Step 2: Excitation (b, c) -> bottleneck -> (b, c)
        y = self.fc(y).view(b, c, 1, 1)
        
        # Step 3: Reweight (broadcast scalar multipliers across entire feature map)
        return x * y.expand_as(x)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Simulating dropping an SEBlock onto the end of a ResNet feature chunk
dummy_se_in = torch.randn(4, 64, 32, 32)
se_module = SEBlock(in_channels=64, reduction=16)

se_out = se_module(dummy_se_in)
print(f"SENet Original Tensor: {dummy_se_in.shape}")
print(f"SENet Reweighted Tensor: {se_out.shape}")
assert se_out.shape == dummy_se_in.shape, "SEBlock must not alter feature dimensions!"


### ⚠️ Common Pitfalls
*   **SE Dimensionality Bugs:** The `Linear` layer expects a flattened vector. Forgetting to `.view(b, c)` before the Linear operation or forgetting to `.view(b, c, 1, 1)` to broadcast afterward will permanently crash your training pipeline.

## 6. DenseNet (All-to-All Skip Connections)

ResNet adds the identity: $L(x) = F(x) + x$. This implies that feature information runs parallel but actively fuses via matrix addition step-by-step. 

DenseNet takes a vastly different philosophy: What if every single layer receives the explicit, concatenated feature maps of **ALL preceding layers**?
Instead of addition, DenseNet explicitly concatenates: $L_{n}(x) = F(x_n, [x_0, x_1, x_2, ..., x_{n-1}])$.

### Mathematical Growth Rate
To make this work, each layer in DenseNet outputs a very tiny number of feature maps, defined as the **Growth Rate ($k$)**. 
Formula Check:
If the initial block holds $C_{in}$ channels, and each layer generates $k$ new channels, the $n$-th layer receives:
$$C_{in} + \sum_{i=1}^{n} (k \times i)$$

DenseNet inherently promotes maximum feature reuse (gradients flow backward instantly to every underlying layer without dilution!)

### DenseNet vs ResNet Analysis
| Architecture | Skip Operator | Parameters | Memory Consumption | Gradient Flow | Implementation Simplicity |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **ResNet** | Element addition (`+`) | Normal | Efficient | Good | High |
| **DenseNet** | Concatenation (`cat`) | **Fewer** | **Huge (Quadratic)** | **Perfect** | Moderate |


In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(DenseLayer, self).__init__()
        # Simplified dense layer (without standard 1x1 BN bottlenecks for clarity)
        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(growth_rate)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Concatenate original input with the newly generated features dimension 1
        new_features = self.relu(self.bn(self.conv(x)))
        return torch.cat([x, new_features], 1)

class MinimalDenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super(MinimalDenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            # Calculate accumulating input connections
            current_in = in_channels + (i * growth_rate)
            layers.append(DenseLayer(current_in, growth_rate))
            
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Initial input: 16 channels
dummy_dense_in = torch.randn(1, 16, 24, 24)

# 4 layers, each growing by k=32
dense_block = MinimalDenseBlock(num_layers=4, in_channels=16, growth_rate=32)
out_dense = dense_block(dummy_dense_in)

print(f"DenseBlock Input Shape: {dummy_dense_in.shape}\n")
print("Printed Proof of Channel Accumulation:")
# After 4 layers with growth_rate=32 and in_channels=16:
# Layer 0 receives: 16. Outputs 32. Total becomes: 16 + 32 = 48.
# Layer 1 receives: 48. Outputs 32. Total becomes: 48 + 32 = 80.
# Layer 2 receives: 80. Outputs 32. Total becomes: 80 + 32 = 112.
# Layer 3 receives: 112. Outputs 32. Total becomes: 112 + 32 = 144.

c = 16
for step in range(4):
    print(f"  Layer {step} receives: {c}. Outputs 32. Total becomes: {c} + 32 = {c + 32}.")
    c += 32

print(f"\nDenseBlock Output Shape: {out_dense.shape}")
assert out_dense.shape == (1, 144, 24, 24), "DenseBlock failed to concatenate growth layers accurately!"


### ⚠️ Common Pitfalls
*   **Assuming Less Params = Fast Training:** Because DenseNets concatenate feature maps massively, intermediate tensor size balloons quadratically. A DenseNet with $40\%$ fewer parameters than a ResNet might still hit Out-Of-Memory errors immediately on a GPU.

### 🎓 Deep Socratic Synthesis

**Q:** *DenseNet and ResNet both use skip connections, but they use them in fundamentally different mathematical ways. DenseNet's memory consumption grows quadratically with depth. What specific engineering trick in DenseNet (involving memory allocation during the forward pass) makes it practical to train despite this growth, and why does this trick NOT work on inference hardware without modification?*

**Ponder deeply:** 
- The trick is **Gradient Checkpointing**! Instead of hoarding every massive concatenation tensor in GPU VRAM during the Forward Pass for backprop, Checkpointing drops the intermediate outputs and instantly recomputes them on-the-fly during the Backward sweep. It trades extra GPU cycles (compute) for massive VRAM savings (memory).
- This trick natively accelerates *training* (which generates a compute graph). At *inference*, where `torch.no_grad()` dominates and backward graphs are not built, memory overhead is purely dictated by standard forward pass concatenation caching, forcing specialized tensor reallocation tricks!
 

## 7. EfficientNet and Compound Scaling

A neural network's power natively depends on scaling three independent spatial characteristics:
1. **Depth ($d$):** Adding more layers.
2. **Width ($w$):** Adding more channels (filters).
3. **Resolution ($r$):** Passing taller/wider imagery formats ($H \times W$).

**The Compound Scaling Law** formalized that scaling ANY single dimension arbitrarily generates heavily diminishing returns. You must scale all three simultaneously using a unified coefficient $\phi$:
- Depth scaling: $d = \alpha^\phi$
- Width scaling: $w = \beta^\phi$
- Resolution scaling: $r = \gamma^\phi$

**The Constraint Equation:** $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$.

**Why is Width and Resolution squared?** 
If you multiply network depth by 2, your FLOPS simply double (Linear). But if you double your input resolution to $2H \times 2W$, you compute exactly $4\times$ more spatial pixels per kernel! Furthermore, if you double your Width (channels), a convolutional operation parameter size is dictated by $(C_{in} \times C_{out})$, triggering a $4\times$ explosion! Hence, width and resolution govern quadratic constraints in the baseline efficiency!

In [ ]:
# Compound Scaling Configuration Utility
EFFICIENTNET_CONSTANTS = {
    # alpha, beta, gamma empirically derived by Google via NAS
    'alpha': 1.2,
    'beta': 1.1,
    'gamma': 1.15
}

def scale_efficientnet(baseline_depth, baseline_width, baseline_res, phi):
    """
    Simulates calculating config dimensions for B0 -> B4.
    """
    a = EFFICIENTNET_CONSTANTS['alpha']
    b = EFFICIENTNET_CONSTANTS['beta']
    g = EFFICIENTNET_CONSTANTS['gamma']
    
    # Validation of the approximate equality constraint
    constraint = a * (b**2) * (g**2)
    assert 1.9 <= constraint <= 2.1, f"Constraint value {constraint:.3f} outside expected range [1.9, 2.1]"
    print(f"Compound scaling constraint: {constraint:.3f} (target ≈ 2.0)")
    
    d = int(baseline_depth * (a ** phi))
    w = int(baseline_width * (b ** phi))
    r = int(baseline_res * (g ** phi))
    
    return d, w, r

print(f"{'Variant':<8} | {'Phi':<4} | {'Depth':<6} | {'Width (Channels)':<18} | {'Resolution'}")
print("-" * 65)

# B0 baseline roughly assumes Depth=16, Channels=32, Resolution=224
for b_variant in range(0, 5):
    d, w, r = scale_efficientnet(baseline_depth=16, baseline_width=32, baseline_res=224, phi=b_variant)
    print(f"B{b_variant:<7} | {b_variant:<4} | {d:<6} | {w:<18} | {r}x{r}")
    
# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# By B4 (phi=4), resolution should scale roughly to 380, width to nearly 47
d4, w4, r4 = scale_efficientnet(16, 32, 224, 4)
assert r4 > 390 and w4 > 45, "Compound scaling mathematical projection failed!"


### ⚠️ Common Pitfalls
*   **Compound Imbalances:** Blindly ramping up image resolution (`800x800`) on a network with a shallow depth topology starves the architecture. Large images require larger Receptive Fields (deep consecutive convolutions) to grasp macroscopic structures.

## Final Core Pitfalls Summary
- **The $1\times1$ oversight:** Avoiding spatial constraints during network expansion causes immediate parameter explosion.
- **ResNet ReLU placement:** Miscalculating Skip-Connection addition order cripples identity routing.
- **DenseNet Memory:** Assuming smaller DenseNet param counts universally means lightweight execution context without accounting for quadratic tensor concat growth!


In [ ]:
# -----------------------------------------------------
# PREVIEW: Learning Rate Schedules for Deep Architectures
# -----------------------------------------------------
def preview_lr_schedule_differences():
    print("Standard CNN Schedule: StepLR (Drop LR by 10x every 30 epochs)")
    print("ViT/Modern Schedule: Linear Warmup -> Cosine Decay")
    print("\nSee NB12 for the full LR scheduler implementation and ViT-specific warmup requirements.")

preview_lr_schedule_differences()
